In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField

KEYS = {"dim_date": "date_key", "dim_route": "route_key",
        "dim_trip": "trip_key", "dim_stop": "stop_key"}

def from_silver(name):
    """Business columns from a silver table, plus _feed_version for lineage."""
    df = spark.table(f"transit.silver.{name}")
    return df.select([c for c in df.columns if not c.startswith("_")] + ["_feed_version"])

def add_hash_key(df, key_name, natural_cols):
    """Surrogate key = 64-bit hash of the natural key. Same input, same key, every run."""
    nulls = df.filter(" OR ".join(f"`{c}` IS NULL" for c in natural_cols)).count()
    assert nulls == 0, f"{key_name}: {nulls} rows with a null natural key"
    return df.select(F.xxhash64(*[F.col(c) for c in natural_cols]).alias(key_name), "*")

def with_unknown(df, key_name, label_cols):
    """Prepend the unknown member: key -1, label columns 'Unknown', everything else null."""
    schema = StructType([StructField(f.name, f.dataType, True) for f in df.schema.fields])
    row = tuple(-1 if c == key_name else ("Unknown" if c in label_cols else None)
                for c in df.columns)
    return spark.createDataFrame([row], schema).unionByName(df)

def check_dim(table, key_name):
    d = spark.table(table)
    n, distinct = d.count(), d.select(key_name).distinct().count()
    unknowns = d.filter(F.col(key_name) == -1).count()
    print(f"{table:24} {n:>9,} rows · keys unique: {n == distinct} · unknown members: {unknowns}")
    assert n == distinct, f"{table}: surrogate key collision or duplicate"
    assert unknowns == 1, f"{table}: expected exactly one unknown member"

def write_gold(df, name):
    (df.withColumn("_gold_loaded_at", F.current_timestamp())
       .write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"transit.gold.{name}"))
    check_dim(f"transit.gold.{name}", KEYS[name])

print("helpers ready")

In [0]:
routes = from_silver("routes").select("route_id")

a = routes.coalesce(1).withColumn("k",  F.monotonically_increasing_id())
b = routes.repartition(8).withColumn("k2", F.monotonically_increasing_id())
cmp = a.join(b, "route_id")

print("routes:                       ", cmp.count())
print("same key on both builds:      ", cmp.filter("k = k2").count())
print("largest key handed out:       ", f"{b.agg(F.max('k2')).collect()[0][0]:,}")
cmp.orderBy("route_id").limit(8).display()

h1 = routes.coalesce(1).withColumn("k",  F.xxhash64("route_id"))
h2 = routes.repartition(8).withColumn("k2", F.xxhash64("route_id"))
print("hash keys identical on both:  ", h1.join(h2, "route_id").filter("k = k2").count())

In [0]:
dates = spark.sql("""
  SELECT explode(sequence(DATE'2024-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS full_date
""")

dim_date = (dates.select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.date_format("full_date", "yyyy-MM").alias("year_month"),
        F.dayofmonth("full_date").alias("day_of_month"),
        (F.expr("weekday(full_date)") + 1).alias("day_of_week"),   # ISO: Monday = 1
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("iso_week"))
    .withColumn("is_weekend", F.col("day_of_week") >= 6))

dim_date = with_unknown(dim_date, "date_key", ["month_name", "year_month", "day_name"])
write_gold(dim_date, "dim_date")

(spark.table("transit.gold.dim_date")
   .filter(F.col("full_date").between("2026-09-14", "2026-09-22"))
   .orderBy("date_key").display())

In [0]:
dim_route = add_hash_key(from_silver("routes"), "route_key", ["route_id"])
dim_route = with_unknown(dim_route, "route_key",
                         ["route_short_name", "route_long_name", "route_type_name"])
write_gold(dim_route, "dim_route")

In [0]:
from pyspark.sql.types import StructType, StructField

trips = from_silver("trips")
labels = [c for c in ["trip_headsign"] if c in trips.columns]

dim_trip = add_hash_key(trips, "trip_key", ["trip_id"])
dim_trip = with_unknown(dim_trip, "trip_key", labels)

# second special member: trips added live, never in any published schedule
nullable = StructType([StructField(f.name, f.dataType, True) for f in dim_trip.schema.fields])
added = spark.createDataFrame(
    [tuple(-2 if c == "trip_key" else ("Added trip (unscheduled)" if c in labels else None)
           for c in dim_trip.columns)], nullable)
dim_trip = dim_trip.unionByName(added)

write_gold(dim_trip, "dim_trip")

In [0]:
stops = from_silver("stops")

parents = stops.select(F.col("stop_id").alias("parent_station"),
                       F.col("stop_name").alias("parent_station_name"))
stops = stops.join(parents, "parent_station", "left")

dim_stop = add_hash_key(stops, "stop_key", ["stop_id"])
dim_stop = with_unknown(dim_stop, "stop_key", ["stop_name", "location_type_name"])
write_gold(dim_stop, "dim_stop")

(spark.table("transit.gold.dim_stop")
   .filter(F.col("parent_station").isNotNull())
   .select("stop_id", "stop_name", "location_type_name", "parent_station", "parent_station_name")
   .limit(10).display())

In [0]:
rebuild = {
    "dim_route": ("routes", "route_key", ["route_id"]),
    "dim_trip":  ("trips",  "trip_key",  ["trip_id"]),
    "dim_stop":  ("stops",  "stop_key",  ["stop_id"]),
}

for name, (src, key, nat) in rebuild.items():
    fresh  = from_silver(src).repartition(8).select(F.xxhash64(*nat).alias("fresh_key"), *nat)
    stored = spark.table(f"transit.gold.{name}").filter(F.col(key) != -1).select(key, *nat)
    moved  = stored.join(fresh, nat).filter(F.col(key) != F.col("fresh_key")).count()
    print(f"{name:10} keys that changed on rebuild: {moved}")
    assert moved == 0, f"{name}: surrogate keys are not stable"

In [0]:
lo, hi = (spark.table("transit.gold.dim_date").filter("date_key != -1")
            .agg(F.min("full_date"), F.max("full_date")).collect()[0])

rt_lo, rt_hi = (spark.table("transit.bronze.rt_vehicle_positions")
                  .agg(F.min("_dt"), F.max("_dt")).collect()[0])

fv = spark.table("transit.silver.stops").select("_feed_version").first()[0]
cal_hi = (spark.table("transit.bronze.gtfs_calendar")
            .filter(F.col("_feed_version") == fv)
            .agg(F.max(F.to_date("end_date", "yyyyMMdd"))).collect()[0][0])

print(f"dim_date covers   {lo} -> {hi}")
print(f"realtime data     {rt_lo} -> {rt_hi}")
print(f"schedule runs to  {cal_hi}")

assert lo <= rt_lo and hi >= rt_hi, "realtime dates fall outside dim_date"
assert hi >= cal_hi, "the schedule runs past the end of dim_date"